# 🏗️ Data Mesh Demo — Overview & Infrastructure Setup

## What is Data Mesh?
Data Mesh is a **decentralized** approach to data architecture where **domain teams own their data as products**. Instead of a central data team bottleneck, each domain is responsible for producing high-quality, discoverable data products within **their own catalog**.

## This Demo
We simulate **two data products** from two independent domains, plus a **federated hub**:

| Catalog | Domain | Source System | Data Product | Owner |
|---------|--------|--------------|--------------|-------|
| `adoit_product` | Enterprise Architecture | **ADOIT** | Application Landscape | EA Team |
| `snow_product` | IT Service Management | **SNOW** (ServiceNow) | Service Health | ITSM Team |
| `data_mesh_hub` | Cross-Domain | Both | Application Risk Profile | Platform Team |

## Architecture — Separate Catalogs per Domain
```
┌─────────────────────────┐  ┌─────────────────────────┐  ┌─────────────────────────┐
│  adoit_product (Catalog) │  │  snow_product (Catalog)  │  │  data_mesh_hub (Catalog) │
│  Owner: EA Team          │  │  Owner: ITSM Team        │  │  Owner: Platform Team     │
├─────────────────────────┤  ├─────────────────────────┤  ├─────────────────────────┤
│ bronze                  │  │ bronze                  │  │ cross_domain            │
│  └ raw_applications     │  │  └ raw_incidents         │  │  └ application_risk_    │
│  └ raw_capabilities     │  │  └ raw_change_requests  │  │      profile             │
│  └ raw_app_cap_map      │  ├─────────────────────────┤  ├─────────────────────────┤
├─────────────────────────┤  │ silver                  │  │ platform                │
│ silver                  │  │  └ incidents             │  │  └ data_quality_log     │
│  └ applications         │  │  └ change_requests      │  └─────────────────────────┘
│  └ business_capabilities│  ├─────────────────────────┤
├─────────────────────────┤  │ gold                    │
│ gold                    │  │  └ service_health ─────┤
│  └ application_      ───┤  └─────────────────────────┘
│      landscape           │         │                     │
└─────────────────────────┘         │                     │
            │                          │                     │
            └────────────────────────┴─────────────────────┘
              Cross-catalog JOINs in data_mesh_hub
```

**Why separate catalogs?** Each domain team has **full autonomy** over their catalog — they manage their own schemas, tables, quality, and access controls. Cross-domain consumption happens via **cross-catalog JOINs** in Unity Catalog, which work seamlessly.

## Data Product Characteristics Mapped to Databricks
| Characteristic | How We Demonstrate It |
|---|---|
| **Clear Owner** | Catalog-level ownership, TBLPROPERTIES (`data_product.owner`), UC Tags |
| **Business Purpose** | Table COMMENT, TBLPROPERTIES (`data_product.business_purpose`) |
| **Defined Consumers** | TBLPROPERTIES (`data_product.consumers`) |
| **Documented Definitions** | Column COMMENTs on every column, table COMMENTs |
| **Quality Checks** | Quality score columns, validation queries, `data_quality_log` table |
| **Access Controls** | GRANT per catalog — domain team owns catalog, consumers get SELECT on gold |
| **Lineage** | Medallion (Bronze→Silver→Gold), cross-catalog lineage via UC |
| **Discoverability** | UC Tags at catalog/schema/table level, Genie Space, Dashboard |
| **Reuse Across Teams** | `data_mesh_hub` joins gold products from both domain catalogs |
| **Reliability / Freshness** | SLA in TBLPROPERTIES, freshness checks in quality log |

## Step 1: Create Domain Catalogs & Schemas
Each domain owns **its own catalog** — true domain autonomy. Schemas follow **Medallion Architecture** (Bronze → Silver → Gold).

In [0]:
%sql
-- ============================================================
-- DOMAIN CATALOG: adoit_product (EA Team owns this entirely)
-- ============================================================
CREATE CATALOG IF NOT EXISTS adoit_product 
MANAGED LOCATION 'abfss://data-products@sadpsddbxdev.dfs.core.windows.net/adoit_product'
COMMENT 'ADOIT Data Product Catalog | Owner: Enterprise Architecture Team | Domain: Enterprise Architecture';

CREATE SCHEMA IF NOT EXISTS adoit_product.bronze COMMENT 'Raw Zone | Ingested from ADOIT EA tool | Owner: EA Team';
CREATE SCHEMA IF NOT EXISTS adoit_product.silver COMMENT 'Curated Zone | Cleansed and enriched | Owner: EA Team';
CREATE SCHEMA IF NOT EXISTS adoit_product.gold   COMMENT 'Product Zone | Business-ready data products | Owner: EA Team';

-- ============================================================
-- DOMAIN CATALOG: snow_product (ITSM Team owns this entirely)
-- ============================================================
CREATE CATALOG IF NOT EXISTS snow_product 
MANAGED LOCATION 'abfss://data-products@sadpsddbxdev.dfs.core.windows.net/snow_product'
COMMENT 'SNOW Data Product Catalog | Owner: IT Service Management Team | Domain: ITSM';

CREATE SCHEMA IF NOT EXISTS snow_product.bronze COMMENT 'Raw Zone | Ingested from ServiceNow | Owner: ITSM Team';
CREATE SCHEMA IF NOT EXISTS snow_product.silver COMMENT 'Curated Zone | Cleansed and enriched | Owner: ITSM Team';
CREATE SCHEMA IF NOT EXISTS snow_product.gold   COMMENT 'Product Zone | Business-ready data products | Owner: ITSM Team';

-- ============================================================
-- HUB CATALOG: data_mesh_hub (Platform Team — cross-domain)
-- ============================================================
CREATE CATALOG IF NOT EXISTS data_mesh_hub 
MANAGED LOCATION 'abfss://data-products@sadpsddbxdev.dfs.core.windows.net/data_mesh_hub'
COMMENT 'Data Mesh Hub | Owner: Data Platform Team | Cross-domain federated products and platform services';

CREATE SCHEMA IF NOT EXISTS data_mesh_hub.cross_domain COMMENT 'Cross-Domain Products | Federated products combining domain catalogs | Owner: Data Platform Team';
CREATE SCHEMA IF NOT EXISTS data_mesh_hub.platform     COMMENT 'Platform Services | Quality monitoring, product registry | Owner: Data Platform Team';

In [0]:
# Clean up dev-prefixed schemas (bundle development mode artifacts)
dev_schemas = [
    'adoit_product.dev_data_mesh_cicd_bronze',
    'adoit_product.dev_data_mesh_cicd_silver',
    'adoit_product.dev_data_mesh_cicd_gold',
    'adoit_product.default',
    'snow_product.dev_data_mesh_cicd_bronze',
    'snow_product.dev_data_mesh_cicd_silver',
    'snow_product.dev_data_mesh_cicd_gold',
    'snow_product.default',
    'data_mesh_hub.dev_data_mesh_cicd_cross_domain',
    'data_mesh_hub.dev_data_mesh_cicd_platform',
    'data_mesh_hub.default',
]
for s in dev_schemas:
    try:
        spark.sql(f'DROP SCHEMA IF EXISTS {s} CASCADE')
        print(f'  Dropped {s}')
    except Exception as e:
        pass  # Schema may not exist
print('✓ Dev schema cleanup complete')


In [0]:
%sql
-- ============================================================
-- Platform tables — schemas match EXACT column usage across notebooks
-- ============================================================

-- Data Product Registry (14 cols: seed INSERT + NB07 reads table_name + NB07 updates)
CREATE OR REPLACE TABLE data_mesh_hub.platform.data_product_registry (
  product_id STRING, product_name STRING, domain STRING, owner_group STRING,
  owner_contact STRING, table_name STRING, status STRING,
  sla_freshness_hours INT, quality_score DOUBLE, freshness_status STRING,
  consumer_count INT, consumers STRING, created_at TIMESTAMP, updated_at TIMESTAMP,
  last_refreshed_at TIMESTAMP, row_count LONG
);

-- Data Contracts (13 cols: matches seed INSERT + NB07/NB08 reads)
CREATE OR REPLACE TABLE data_mesh_hub.platform.data_contracts (
  contract_id STRING, product_id STRING, product_name STRING, producer_group STRING,
  consumer_group STRING, contract_status STRING, agreed_sla_hours INT,
  quality_threshold_pct DOUBLE, schema_version STRING, retention_days INT,
  escalation_contact STRING, escalation_channel STRING, created_at TIMESTAMP
);

-- Data Quality Log (12 cols: matches NB04 INSERT exactly)
CREATE OR REPLACE TABLE data_mesh_hub.platform.data_quality_log (
  check_id STRING, check_timestamp TIMESTAMP, data_product STRING,
  domain STRING, table_reference STRING, check_name STRING,
  check_type STRING, description STRING, passed BOOLEAN,
  expected_value STRING, actual_value STRING, severity STRING
);

-- Product Observability (16 cols: matches NB07 Cell 12 INSERT VALUES)
CREATE OR REPLACE TABLE data_mesh_hub.platform.product_observability (
  obs_id STRING, product_id STRING, product_name STRING, check_timestamp TIMESTAMP,
  row_count LONG, row_count_change LONG, freshness_hours DOUBLE, sla_met BOOLEAN,
  quality_checks_total INT, quality_checks_passed INT, quality_score DOUBLE,
  column_count INT, schema_drift BOOLEAN, active_contracts INT,
  consumer_count INT, status STRING
);

-- Domain Maturity Scorecard (14 cols: matches seed INSERT)
CREATE OR REPLACE TABLE data_mesh_hub.platform.domain_maturity_scorecard (
  domain STRING, maturity_level STRING, overall_maturity_score INT,
  data_quality_score INT, accessibility_score INT, documentation_score INT,
  interoperability_score INT, governance_score INT, security_score INT,
  observability_score INT, self_serve_score INT, product_count INT,
  consumer_count INT, uptime_pct DOUBLE
);

-- Cross-domain product table (22 cols: matches NB03 MERGE source)
CREATE OR REPLACE TABLE data_mesh_hub.cross_domain.application_risk_profile (
  app_id STRING, app_name STRING, app_owner STRING, department STRING,
  criticality STRING, lifecycle_status STRING, technology_stack STRING,
  is_cloud_hosted BOOLEAN, app_age_years DOUBLE, capability_count INT,
  total_incidents INT, open_incidents INT, critical_incidents INT,
  sla_breaches INT, sla_compliance_pct DOUBLE, avg_resolution_hours DOUBLE,
  total_changes INT, change_success_rate_pct DOUBLE, service_risk_score STRING,
  composite_risk_score STRING, risk_factors STRING, product_generated_at TIMESTAMP
);

In [0]:
# ── Load platform seed data ──
import os

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
bundle_root = notebook_path.rsplit('/src/notebooks/', 1)[0]
seed_path = f"/Workspace{bundle_root}/src/data/platform_seed.sql"

if not os.path.exists(seed_path):
    seed_path = "/Workspace/Users/skarotirajashekar@godevsuite060.onmicrosoft.com/data-mesh-demo/src/data/platform_seed.sql"

with open(seed_path) as f:
    seed_sql = f.read()

# Execute each statement separately (strip comment lines before checking)
for stmt in seed_sql.split(';'):
    # Remove comment-only lines to get the actual SQL
    sql_lines = [l for l in stmt.strip().split('\n') if l.strip() and not l.strip().startswith('--')]
    sql = '\n'.join(sql_lines).strip()
    if sql:
        try:
            spark.sql(sql)
            # Show what was executed
            first_kw = sql.split()[0:4]
            print(f"  ✓ {' '.join(first_kw)}...")
        except Exception as e:
            print(f"  ⚠ {str(e)[:120]}")

print("\n✓ Platform seed data loaded (registry, contracts, maturity scorecard)")

## Step 2: Tag Catalogs, Schemas & Tables for Discoverability
Unity Catalog **Tags** at every level — catalog, schema, and table — make assets discoverable via search and Catalog Explorer.

In [0]:
%sql
-- Tag CATALOGS with domain ownership and mesh role
ALTER CATALOG adoit_product SET TAGS ('domain' = 'Enterprise Architecture', 'owner' = 'EA Team', 'data_mesh_role' = 'Domain Catalog');
ALTER CATALOG snow_product  SET TAGS ('domain' = 'IT Service Management', 'owner' = 'ITSM Team', 'data_mesh_role' = 'Domain Catalog');
ALTER CATALOG data_mesh_hub SET TAGS ('domain' = 'Cross-Domain', 'owner' = 'Data Platform Team', 'data_mesh_role' = 'Hub Catalog');

-- Tag SCHEMAS with domain, owner, and tier
ALTER SCHEMA adoit_product.bronze SET TAGS ('domain' = 'Enterprise Architecture', 'owner' = 'EA Team', 'tier' = 'bronze');
ALTER SCHEMA adoit_product.silver SET TAGS ('domain' = 'Enterprise Architecture', 'owner' = 'EA Team', 'tier' = 'silver');
ALTER SCHEMA adoit_product.gold   SET TAGS ('domain' = 'Enterprise Architecture', 'owner' = 'EA Team', 'tier' = 'gold');
ALTER SCHEMA snow_product.bronze  SET TAGS ('domain' = 'IT Service Management', 'owner' = 'ITSM Team', 'tier' = 'bronze');
ALTER SCHEMA snow_product.silver  SET TAGS ('domain' = 'IT Service Management', 'owner' = 'ITSM Team', 'tier' = 'silver');
ALTER SCHEMA snow_product.gold    SET TAGS ('domain' = 'IT Service Management', 'owner' = 'ITSM Team', 'tier' = 'gold');
ALTER SCHEMA data_mesh_hub.cross_domain SET TAGS ('domain' = 'Cross-Domain', 'owner' = 'Data Platform Team', 'tier' = 'gold');
ALTER SCHEMA data_mesh_hub.platform     SET TAGS ('domain' = 'Platform', 'owner' = 'Data Platform Team', 'tier' = 'platform');

-- NOTE: Gold table tagging is done in 08_Advanced_Governance (after tables are created by pipeline notebooks)

## Step 3: Access Control Patterns
With **separate catalogs**, each domain team has **full autonomy** over their access controls:

```sql
-- ============================================================
-- EA TEAM: Full control of their adoit_product catalog
-- ============================================================
GRANT ALL PRIVILEGES ON CATALOG adoit_product TO `ea-team`;

-- Consumers get READ-ONLY on the gold data products
GRANT USE CATALOG  ON CATALOG adoit_product TO `cto-office`;
GRANT USE SCHEMA   ON SCHEMA  adoit_product.gold TO `cto-office`;
GRANT SELECT       ON TABLE   adoit_product.gold.application_landscape TO `cto-office`;

-- ============================================================
-- ITSM TEAM: Full control of their snow_product catalog
-- ============================================================
GRANT ALL PRIVILEGES ON CATALOG snow_product TO `itsm-team`;

-- Consumers get READ-ONLY on gold
GRANT USE CATALOG  ON CATALOG snow_product TO `risk-team`;
GRANT USE SCHEMA   ON SCHEMA  snow_product.gold TO `risk-team`;
GRANT SELECT       ON TABLE   snow_product.gold.service_health TO `risk-team`;

-- ============================================================
-- PLATFORM TEAM: Manages the hub, grants read to everyone
-- ============================================================
GRANT ALL PRIVILEGES ON CATALOG data_mesh_hub TO `data-platform-team`;

GRANT USE CATALOG ON CATALOG data_mesh_hub TO `cto-office`;
GRANT USE CATALOG ON CATALOG data_mesh_hub TO `risk-team`;
GRANT SELECT ON TABLE data_mesh_hub.cross_domain.application_risk_profile TO `cto-office`;
GRANT SELECT ON TABLE data_mesh_hub.cross_domain.application_risk_profile TO `risk-team`;
```

**Key difference from single-catalog**: The EA Team can manage `adoit_product` access without involving the Platform Team. True domain autonomy.

## Step 4: Inspect the Data Product Metadata
Data product properties are embedded in Unity Catalog, making them discoverable via `DESCRIBE EXTENDED` or the Catalog Explorer UI.

In [0]:
%sql
-- Verify catalogs and schemas created successfully
SHOW SCHEMAS IN adoit_product;

In [0]:
%sql
-- Verify SNOW schemas
SHOW SCHEMAS IN snow_product;

In [0]:
%sql
-- Verify Hub schemas
SHOW SCHEMAS IN data_mesh_hub;

## Next Steps
Run the domain-specific notebooks to see the full pipeline in action:
- **01_ADOIT_Data_Product_Pipeline** — Bronze→Silver→Gold within `adoit_product` catalog
- **02_SNOW_Data_Product_Pipeline** — Bronze→Silver→Gold within `snow_product` catalog
- **03_Cross_Domain_Data_Product** — Cross-catalog federated consumption in `data_mesh_hub`
- **04_Data_Quality_Monitoring** — Quality checks across all 3 catalogs

In [0]:
%sql
-- Grant read access to all workspace users on data mesh catalogs
GRANT USE CATALOG ON CATALOG adoit_product TO `account users`;
GRANT USE SCHEMA ON CATALOG adoit_product TO `account users`;
GRANT SELECT ON CATALOG adoit_product TO `account users`;

GRANT USE CATALOG ON CATALOG snow_product TO `account users`;
GRANT USE SCHEMA ON CATALOG snow_product TO `account users`;
GRANT SELECT ON CATALOG snow_product TO `account users`;

GRANT USE CATALOG ON CATALOG data_mesh_hub TO `account users`;
GRANT USE SCHEMA ON CATALOG data_mesh_hub TO `account users`;
GRANT SELECT ON CATALOG data_mesh_hub TO `account users`;
